In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import os 
from thefuzz import process
import pandas as pd
import os
import re
import holidays
import tkinter as tk
from tkinter import filedialog
from collections import defaultdict
from clase_reportes_new import ReportClassNew
import json
from collections import defaultdict
from send_mail import MailSender
import pandas as pd
ms = MailSender()
rc = ReportClassNew()

In [ ]:
ruta = rc.validar_ruta()
ruta_pcn = ruta / 'data' / 'contabilidad' / 'balance_comprobacion' / 'pcn'
# ruta_hfa = ruta / 'data' / 'contabilidad' / 'balance_comprobacion' / 'hfa'
rutas = [
    # ruta_hfa,
          ruta_pcn
          ]

meses = {
    'ene': 1, 'feb': 2, 'mar': 3, 'abr': 4, 'may': 5, 'jun': 6,
    'jul': 7, 'ago': 8, 'sep': 9, 'oct': 10, 'nov': 11, 'dic': 12
}

dfs = []

for ruta_carpeta in rutas:
    try:    
        for archivo in os.listdir(ruta_carpeta):
            if archivo.endswith(f'.{'xlsx'}'):
                ruta_completa = os.path.join(ruta_carpeta, archivo)
                df_balance= pd.read_excel(ruta_completa)

                # Crear fecha de informe desde el nombre de columna
                valor = df_balance.columns[4]  # ej: 'ene 2026'
                mes_txt, anio = valor.split()
                fecha_informe = pd.Timestamp(int(anio), meses[mes_txt.lower()], 1)

                # Usar fila como encabezado
                df_balance.columns = df_balance.iloc[1].values

                # Renombrar primeras columnas
                cols = list(df_balance.columns)
                cols[0] = 'cuenta'
                cols[1] = 'descripcion'
                cols[2] = 'debe_inicial'
                cols[3] = 'haber_inicial'

                cols[4] = 'debe'
                cols[5] = 'haber'

                cols[6] = 'debe_final'
                cols[7] = 'haber_final'

                df_balance.columns = cols
                print(df_balance.columns)
                # Filtrar filas válidas
                df_balance = df_balance[df_balance['cuenta'].notna()]

                # # Elimina las columnas de solo NaN
                # df_balance = df_balance.dropna(axis=1, how='all')

                # Empresa según carpeta
                if ruta_carpeta.name == 'hfa':
                    df_balance['empresa'] = 'hfa'
                elif ruta_carpeta.name == 'pcn':
                    df_balance['empresa'] = 'pcn'
                else:
                    df_balance['empresa'] = 'desconocido'

                # Agregar fecha
                df_balance['fecha_informe'] = fecha_informe


                df_balance['cuenta'] = df_balance['cuenta'].astype(int)

             

                dfs.append(df_balance)

    except Exception as e:
        print(f"Error al procesar la info en {ruta_carpeta}: {e}")

Index(['cuenta', 'descripcion', 'debe_inicial', 'haber_inicial', 'debe',
       'haber', 'debe_final', 'haber_final'],
      dtype='object')
Index(['cuenta', 'descripcion', 'debe_inicial', 'haber_inicial', 'debe',
       'haber', 'debe_final', 'haber_final'],
      dtype='object')


In [3]:
df_consolidado = pd.concat(dfs, ignore_index=True, sort=False)

In [4]:
# crear niveles contables
df_consolidado['nivel_1'] = df_consolidado['cuenta'].astype(str).str[:1]
df_consolidado['nivel_2'] = df_consolidado['cuenta'].astype(str).str[:2]
df_consolidado['nivel_3'] = df_consolidado['cuenta'].astype(str).str[:4]
df_consolidado['nivel_4'] = df_consolidado['cuenta'].astype(str).str[:6]
df_consolidado['nivel_5'] = df_consolidado['cuenta'].astype(str).str[:8]

In [5]:
map_naturaleza = {
    '1': 'deudora',   # Activo
    '5': 'deudora',   # Gastos
    '6': 'deudora',   # Costos
    '2': 'acreedora', # Pasivo
    '3': 'acreedora', # Patrimonio
    '4': 'acreedora'  # Ingresos
}
df_consolidado['naturaleza'] = df_consolidado['nivel_1'].map(map_naturaleza)

In [6]:
df_consolidado['saldo_inicial'] = np.where(
    df_consolidado['naturaleza'] == 'deudora',
    df_consolidado['debe_inicial'] - df_consolidado['haber_inicial'],
    df_consolidado['haber_inicial'] - df_consolidado['debe_inicial']
)

In [7]:
df_consolidado['saldo_final'] = np.where(
    df_consolidado['naturaleza'] == 'deudora',
    df_consolidado['debe_final'] - df_consolidado['haber_final'],
    df_consolidado['haber_final'] - df_consolidado['debe_final']
)

In [8]:
df_consolidado['saldo_calculado'] = df_consolidado['saldo_inicial'] + (
    df_consolidado['debe'] - df_consolidado['haber']
)

In [9]:
df_consolidado.columns

Index(['cuenta', 'descripcion', 'debe_inicial', 'haber_inicial', 'debe',
       'haber', 'debe_final', 'haber_final', 'empresa', 'fecha_informe',
       'nivel_1', 'nivel_2', 'nivel_3', 'nivel_4', 'nivel_5', 'naturaleza',
       'saldo_inicial', 'saldo_final', 'saldo_calculado'],
      dtype='object')

In [10]:

df_consolidado.loc[df_consolidado['nivel_1'] == '1', 'Valor'] = df_consolidado['saldo_final']

df_consolidado.loc[
    df_consolidado['nivel_1'].isin(['2', '3']),
    'Valor'
] = df_consolidado['saldo_final'] * -1

df_consolidado.loc[
    df_consolidado['nivel_1'] == '4',
    'Valor'
] = df_consolidado['haber'] - df_consolidado['debe']

df_consolidado.loc[
    df_consolidado['nivel_1'].isin(['5', '6', '7']),
    'Valor'
] = df_consolidado['debe'] - df_consolidado['haber']

In [11]:
df_consolidado.to_excel(r'D:\Desktop\'bc.xlsx', index=False)